# MedSAM2: segment a 3D CT by prompting a single slice

SAM 2 tracks objects across video frames using memory attention. A CT volume has the
same structure — consecutive slices barely differ — so one box on one slice can be
propagated through the whole stack.

**Runtime → Change runtime type → T4 GPU** before running anything.

- MedSAM2: https://github.com/bowang-lab/MedSAM2 · weights research/education only
- Paper: https://arxiv.org/abs/2504.03600

## 1 · Setup

In [ ]:
import torch

assert torch.cuda.is_available(), "Runtime -> Change runtime type -> T4 GPU, then rerun."
print(torch.cuda.get_device_name(0))

In [ ]:
%%capture
!pip install -q git+https://github.com/rekalantar/medsam2-3d-ct.git
!pip install -q SimpleITK imageio huggingface_hub
!git clone -q https://github.com/bowang-lab/MedSAM2.git /content/MedSAM2
%cd /content/MedSAM2
!pip install -q -e ".[dev]"
!bash download.sh

In [ ]:
import os

import numpy as np
import matplotlib.pyplot as plt
from huggingface_hub import hf_hub_download

from medsam2_ct import (build_predictor, init_state, largest_component,
                        load_volume, plot_slices, save_gif, segment_volume,
                        window_hu)

CKPT = "/content/MedSAM2/checkpoints/MedSAM2_latest.pt"
assert os.path.exists(CKPT), "download.sh did not produce MedSAM2_latest.pt"

DATASET = "wanglab/CT_DeepLesion-MedSAM2"
ROTATE = 1      # clockwise quarter-turns, display only — these volumes read sideways

predictor = build_predictor(config="configs/sam2.1_hiera_t512.yaml", checkpoint=CKPT)
print(f"checkpoint {os.path.getsize(CKPT) / 1e6:.0f} MB — ready")

## 2 · Load a case

The demo dataset ships per-case volumes with ground-truth masks, a few MB each.
Having the label lets us place the prompt automatically and score the result.

In [ ]:
def dice(a, b):
    a, b = np.asarray(a, bool), np.asarray(b, bool)
    total = a.sum() + b.sum()
    return 1.0 if total == 0 else 2 * (a & b).sum() / total


def load_case(case, margin=5):
    """Fetch, window, and place a box on the largest lesion cross-section."""
    image = hf_hub_download(DATASET, f"images/{case}_0000.nii.gz", repo_type="dataset")
    label = hf_hub_download(DATASET, f"labels/{case}.nii.gz", repo_type="dataset")

    volume_hu, spacing = load_volume(image)
    truth = load_volume(label)[0] > 0

    key = int(truth.sum(axis=(1, 2)).argmax())
    ys, xs = np.where(truth[key])
    box = [int(xs.min()) - margin, int(ys.min()) - margin,
           int(xs.max()) + margin, int(ys.max()) + margin]

    return dict(name=case, hu=volume_hu, volume=window_hu(volume_hu, 400, 40),
                truth=truth, spacing=spacing, key=key, box=box,
                span=int(truth.any(axis=(1, 2)).sum()))


case = load_case("000009_03_01_036-048")
print(f"volume  {case['volume'].shape}  spacing {tuple(round(s, 2) for s in case['spacing'])}")
print(f"HU      {case['hu'].min():.0f} to {case['hu'].max():.0f}")
print(f"lesion  {case['span']} slices, largest at {case['key']}")

CT is stored in Hounsfield units across thousands of values; the encoder takes 8-bit.
`window_hu(volume, 400, 40)` keeps the abdominal soft-tissue range and discards the
rest, so the lesion gets the full 256 levels instead of a handful.

Worth checking the HU range printed above: if it already spans only a few hundred, the
volume arrived pre-clipped and windowing is a no-op on it.

## 3 · Prompt one slice

In [ ]:
import matplotlib.patches as patches
from medsam2_ct import rotate_clockwise

key, box = case["key"], case["box"]
shown = rotate_clockwise(case["volume"], ROTATE)[key]

fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))
axes[0].imshow(shown, cmap="gray")
axes[0].set_title(f"slice {key}")

axes[1].imshow(shown, cmap="gray")
axes[1].contour(rotate_clockwise(case["truth"], ROTATE)[key], levels=[0.5],
                colors="#3FC1C9", linewidths=1.5)
axes[1].set_title("ground truth (cyan)")
for ax in axes: ax.axis("off")
plt.tight_layout()

print(f"box {box}  (in the volume's own pixel coordinates, pre-rotation)")

## 4 · Propagate

`segment_volume` sweeps forward from the prompted slice and then backward. That second
sweep looks redundant — it isn't. The prompted slice sits in the middle of the lesion,
so forward-only stops halfway.

In [ ]:
masks = largest_component(segment_volume(predictor, case["volume"], box, key))

truth = case["truth"]
print(f"prediction    {masks.sum():>7,} voxels   Dice {dice(truth, masks):.3f}")
print(f"ground truth  {truth.sum():>7,} voxels")
print(f"key slice                        Dice {dice(truth[key], masks[key]):.3f}")

**What the second sweep is worth.** Same prompt, forward only:

In [ ]:
state, _, _ = init_state(predictor, case["volume"])
predictor.add_new_points_or_box(
    inference_state=state, frame_idx=key, obj_id=1,
    box=np.asarray(box, dtype=np.float32))

forward = np.zeros_like(masks)
for idx, _ids, logits in predictor.propagate_in_video(state):
    forward[idx] = (logits[0] > 0).cpu().numpy().squeeze()

print(f"both ways     {masks.sum():>7,} voxels   Dice {dice(truth, masks):.3f}")
print(f"forward only  {forward.sum():>7,} voxels   Dice {dice(truth, forward):.3f}"
      f"   ({forward.sum() / masks.sum():.0%})")

## 5 · Look at it

Every slice the lesion touches, cropped and rotated to read the way an axial slice
should. Cyan is ground truth, coral is the prediction.

In [ ]:
fig = plot_slices(case["volume"], masks, truth, rotate=ROTATE, ncols=7,
                  title=f"{case['name']} — Dice {dice(truth, masks):.3f}")
fig.savefig("slices.png", dpi=150, bbox_inches="tight")

And the same thing as an animation, which is where propagation failures show up.

In [ ]:
z = (masks | truth).any(axis=(1, 2)).nonzero()[0]
lo, hi = max(0, z.min() - 2), min(len(case["volume"]), z.max() + 3)

save_gif(case["volume"][lo:hi], masks[lo:hi], "propagation.gif",
         fps=4, rotate=ROTATE)
print(f"{os.path.getsize('propagation.gif') / 1e6:.2f} MB, {hi - lo} frames")

In [ ]:
import IPython.display

IPython.display.Image("propagation.gif")

## 6 · Where the error lives

Volume Dice is one number over a structure whose slices are not equally hard. Scoring
each slice against its distance from the prompt shows where the cost actually falls.

In [ ]:
def per_slice_dice(c, m):
    z = c["truth"].any(axis=(1, 2)).nonzero()[0]
    return z - c["key"], np.array([dice(c["truth"][i], m[i]) for i in z])


offsets, scores = per_slice_dice(case, masks)
for off, d in zip(offsets, scores):
    print(f"  {off:+3d}  {d:.3f}  {'#' * int(d * 40)}")

plt.figure(figsize=(8, 3.5))
plt.plot(offsets, scores, "o-", color="#3FC1C9", linewidth=2)
plt.axvline(0, color="#FF6B5B", linestyle="--", label="prompted slice")
plt.xlabel("slices from prompt"); plt.ylabel("Dice")
plt.ylim(0, 1.05); plt.legend(); plt.tight_layout()
plt.savefig("decay.png", dpi=150, bbox_inches="tight")

## 7 · A longer lesion

The case above spans a handful of slices, so propagation never travelled far. This one
spans many more, which is the real test of whether the method holds up.

In [ ]:
case2 = load_case("000002_02_01_044-083")
masks2 = largest_component(
    segment_volume(predictor, case2["volume"], case2["box"], case2["key"]))

for c, m in ((case, masks), (case2, masks2)):
    off, sc = per_slice_dice(c, m)
    print(f"{c['name']}")
    print(f"   {c['span']:>3} slices   volume Dice {dice(c['truth'], m):.3f}"
          f"   key slice {dice(c['truth'][c['key']], m[c['key']]):.3f}")
    print(f"   best slice at offset {off[sc.argmax()]:+d}")
    print()

In [ ]:
off2, sc2 = per_slice_dice(case2, masks2)

plt.figure(figsize=(9, 3.5))
plt.plot(offsets, scores, "o-", color="#3FC1C9", linewidth=2,
         label=f"{case['span']} slices")
plt.plot(off2, sc2, "o-", color="#FF6B5B", linewidth=2,
         label=f"{case2['span']} slices")
plt.axvline(0, color="#999", linestyle="--")
plt.xlabel("slices from prompt"); plt.ylabel("Dice")
plt.ylim(0, 1.05); plt.legend(); plt.tight_layout()
plt.savefig("decay_both.png", dpi=150, bbox_inches="tight")

In [ ]:
fig = plot_slices(case2["volume"], masks2, case2["truth"], rotate=ROTATE, ncols=8,
                  title=f"{case2['name']} — Dice {dice(case2['truth'], masks2):.3f}")
fig.savefig("slices_long.png", dpi=150, bbox_inches="tight")

---

Project: https://github.com/rekalantar/medsam2-3d-ct